# Generación de datos bancarios sintéticos para análisis de riesgo de crédito

La meta del presente apartado es crear **11 tablas CSV relacionadas** donde estaremos simulando la información asociada a los clientes de un banco.

**Corte temporal:** historial previo hasta $2025/10/31$, solicitudes $2025/11/01$ y resultados hasta $2026/11/01$.

La etiqueta objetivo es un atraso de al menos $90$ días dentro de los $12$ meses posteriores a la solicitud, únicamente en préstamos aprobados.



In [5]:
# Importando las bibliotecas que serán utilizadas.
import json
import math
import zipfile
import hashlib
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd

Primero, se establecen los parámetros generales, como el número de clientes, la moneda, las fechas de análisis y los criterios de aprobación de préstamos. También se definien las funciones auxiliares para calcular cuotas, registrar transacciones y organizar los datos.

In [6]:
N_CLIENTES = 1000       # Aumentar a 5000+ para experimentos de ML mas representativos.
SEMILLA = 2026
BANCO_REFERENCIA = 'BBVA Espana (SIMULACION ACADEMICA)'
MONEDA = 'EUR'
FECHA_INICIO = pd.Timestamp('2023-01-01')
FECHA_FIN_HISTORIA = pd.Timestamp('2026-10-31')
FECHA_CORTE_RESULTADOS = pd.Timestamp('2026-11-01')
MESES = pd.date_range('2023-01-01', '2026-10-01', freq='MS')
FECHAS_COHORTES = pd.to_datetime([
    '2024-08-01', '2024-11-01', '2025-02-01', '2025-05-01',
    '2025-08-01', '2025-11-01', '2026-02-01', '2026-05-01', '2026-08-01'
])
ESCENARIO_DRIFT_2026 = True  # Aumento simulado de gastos y uso digital desde enero de 2026.
CARPETA_SALIDA = Path('/content/dataset_bbva_sintetico' if Path('/content').exists() else '/mnt/data/dataset_bbva_sintetico_v2')
CARPETA_SALIDA.mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(SEMILLA)

# Criterios didacticos, NO criterios oficiales del banco.
POLITICA = {
    'id': 'POL-SIM-01', 'version': '1.0', 'ingreso_minimo_eur': 1200,
    'dtimax': 0.40, 'margen_libre_minimo_eur': 150,
    'antiguedad_laboral_minima_meses': 6, 'historial_bancario_minimo_meses': 6,
    'mora_previa_rechazo_dias': 60, 'importe_min_eur': 3000,
    'importe_max_eur': 75000, 'plazos_meses': [24, 36, 48, 60, 72, 84, 96]
}

NOMBRES = ['Ana','Luis','Maria','Jose','Carolina','Pedro','Sofia','Miguel','Laura','Daniel',
           'Carmen','Javier','Isabel','Andres','Elena','Rafael','Patricia','Manuel','Valeria','Carlos']
APELLIDOS = ['Garcia','Rodriguez','Martinez','Perez','Lopez','Fernandez','Sanchez','Gomez',
             'Ramirez','Torres','Diaz','Castillo','Ortiz','Reyes','Mendoza','Herrera','Vargas']
PROVINCIAS = ['Madrid', 'Barcelona', 'Valencia', 'Sevilla', 'Malaga', 'Zaragoza', 'Bilbao']
CATEGORIAS_GASTOS = ['VIVIENDA', 'ALIMENTACION', 'TRANSPORTE', 'SUMINISTROS',
                      'SALUD', 'EDUCACION', 'COMERCIO', 'OCIO']

clientes, cuentas, tarjetas = [], [], []
transacciones, accesos, saldos_mensuales, estados_tarjetas = [], [], [], []
movimientos_tarjetas, panel_cliente_mensual = [], []
interacciones_crm, encuestas, contactos_campana = [], [], []
historial_crediticio, solicitudes, snapshots, pagos_prestamos = [], [], [], []
metricas_operativas = []
perfil_por_cliente, cuentas_por_cliente, tarjetas_credito_por_cliente = {}, {}, {}
saldo_por_cuenta, deuda_por_tarjeta = {}, {}
num_tx = num_acc = num_mov_tarjeta = num_traspaso = 0


def es_cliente_activo(perfil, fecha):
    return fecha >= perfil['fecha_alta'] and (pd.isna(perfil['fecha_baja']) or fecha < perfil['fecha_baja'])


def registrar_transaccion(cliente_id, cuenta_id, fecha, tipo, categoria, importe, traspaso_id=None):
    """Actualiza el saldo y registra un movimiento. Prohibe saldo negativo."""
    global num_tx
    importe = round(max(float(importe), 0), 2)
    if tipo in ('EGRESO', 'TRANSFERENCIA_SALIDA'):
        importe = round(min(importe, saldo_por_cuenta[cuenta_id]), 2)
    if importe < 0.01:
        return 0.0
    entrada = tipo in ('INGRESO', 'TRANSFERENCIA_ENTRADA')
    saldo_por_cuenta[cuenta_id] = round(saldo_por_cuenta[cuenta_id] + (importe if entrada else -importe), 2)
    num_tx += 1
    transacciones.append({
        'transaccion_id': f'MOV{num_tx:09d}', 'cliente_id': cliente_id,
        'cuenta_id': cuenta_id, 'fecha': fecha, 'tipo': tipo, 'categoria': categoria,
        'direccion': 'ENTRADA' if entrada else 'SALIDA', 'importe_eur': importe,
        'saldo_posterior_eur': saldo_por_cuenta[cuenta_id], 'traspaso_id': traspaso_id
    })
    return importe


def traspasar(cliente_id, origen, destino, fecha, importe):
    """Registra ambas caras de un traspaso sin contarlo como renta ni consumo."""
    global num_traspaso
    num_traspaso += 1
    clave = f'TRF{num_traspaso:08d}'
    movido = registrar_transaccion(cliente_id, origen, fecha, 'TRANSFERENCIA_SALIDA',
                                   'TRASPASO_PROPIO', importe, clave)
    if movido:
        registrar_transaccion(cliente_id, destino, fecha, 'TRANSFERENCIA_ENTRADA',
                              'TRASPASO_PROPIO', movido, clave)


def registrar_mov_tarjeta(cliente_id, tarjeta_id, fecha, tipo, importe):
    global num_mov_tarjeta
    num_mov_tarjeta += 1
    movimientos_tarjetas.append({
        'movimiento_tarjeta_id': f'MTC{num_mov_tarjeta:09d}',
        'cliente_id': cliente_id, 'tarjeta_id': tarjeta_id, 'fecha': fecha,
        'tipo': tipo, 'importe_eur': round(float(importe), 2),
        'saldo_utilizado_posterior_eur': round(deuda_por_tarjeta[tarjeta_id], 2)
    })


def cuota_francesa(importe, tin_anual, n_meses):
    r = float(tin_anual) / 12
    return round(importe * r / (1 - (1 + r)**(-n_meses)), 2) if r else round(importe / n_meses, 2)


def particion_temporal(fecha):
    if fecha <= pd.Timestamp('2025-02-01'):
        return 'TRAIN'
    if fecha == pd.Timestamp('2025-05-01'):
        return 'VALIDACION'
    if fecha <= pd.Timestamp('2025-11-01'):
        return 'TEST_TEMPORAL'
    return 'MONITORIZACION_SIN_ETIQUETA_MADURA'

Primero, vamos a crear los clientes con sus datos personales, incluyendo información personal, situación laboral, ingresos mensuales y antigüedad bancaria. A cada cliente se le asignan cuentas y tarjetas de débito o crédito, estableciendo sus saldos iniciales, límites de crédito y características financieras.

In [7]:
for i in range(1, N_CLIENTES + 1):
    cli = f'CLI{i:06d}'
    alta = (pd.Timestamp('2018-01-01') + pd.DateOffset(
        months=int(rng.integers(0, 60 if rng.random() < 0.68 else 88))))
    # La muestra incluye algunas bajas; el banco puede estudiar retencion.
    baja = (pd.Timestamp('2025-01-01') + pd.DateOffset(months=int(rng.integers(0, 22)))) if rng.random() < 0.045 else pd.NaT
    if pd.notna(baja) and baja <= alta:
        baja = pd.NaT
    ocupacion = str(rng.choice(['ASALARIADO','AUTONOMO','TEMPORAL','PENSIONISTA'], p=[.59,.22,.11,.08]))
    ingreso_base = round(float(np.clip(rng.lognormal(np.log(2100), 0.49), 850, 9200)), 2)
    riesgo_latente = float(rng.beta(2.0, 4.2))  # Nunca se exporta como predictor.
    volatilidad = {'ASALARIADO': .045, 'AUTONOMO': .22, 'TEMPORAL': .28, 'PENSIONISTA': .018}[ocupacion]
    edad = int(rng.integers(22, 73))
    fecha_nacimiento = pd.Timestamp('2026-01-01') - pd.DateOffset(years=edad, days=int(rng.integers(0, 350)))
    inicio_empleo = alta - pd.DateOffset(months=int(rng.integers(-4, 110)))
    num_cuentas = int(rng.choice([1,2,3], p=[.66,.29,.05]))
    cuentas_por_cliente[cli], tarjetas_credito_por_cliente[cli] = [], []
    n_debito, n_credito = 0, 0
    for j in range(1, num_cuentas + 1):
        cid = f'CTA{i:06d}{j}'
        saldo_inicial = round(float(ingreso_base * rng.uniform(.12, 1.4) / np.sqrt(num_cuentas)), 2)
        saldo_por_cuenta[cid] = saldo_inicial
        cuentas.append({'cuenta_id': cid, 'cliente_id': cli,
                        'tipo_cuenta': 'CORRIENTE' if j == 1 else 'AHORRO',
                        'fecha_apertura': alta, 'moneda': MONEDA,
                        'saldo_inicial_eur': saldo_inicial})
        cuentas_por_cliente[cli].append(cid)
        if rng.random() < .89:
            tid = f'TDB{i:06d}{j}'
            tarjetas.append({'tarjeta_id': tid, 'cliente_id': cli, 'cuenta_id': cid,
                             'tipo_tarjeta': 'DEBITO', 'limite_credito_eur': np.nan})
            n_debito += 1
    n_credito = int(rng.choice([0,1,2], p=[.36,.55,.09]))
    for j in range(1, n_credito + 1):
        tid = f'TCR{i:06d}{j}'
        limite = round(float(np.clip(ingreso_base * rng.uniform(.4, 3.3) / np.sqrt(n_credito), 500, 14000)), 2)
        deuda_por_tarjeta[tid] = round(limite * float(np.clip(riesgo_latente + rng.normal(0, .12), .01, .9)), 2)
        tarjetas_credito_por_cliente[cli].append((tid, limite))
        tarjetas.append({'tarjeta_id': tid, 'cliente_id': cli, 'cuenta_id': None,
                         'tipo_tarjeta': 'CREDITO', 'limite_credito_eur': limite})
    consentimiento = bool(rng.random() < .70)
    clientes.append({
        'cliente_id': cli,
        'nombre_completo': f'{rng.choice(NOMBRES)} {rng.choice(APELLIDOS)} {rng.choice(APELLIDOS)}',
        'telefono_ficticio': f'TEL-SIM-{i:07d}', 'dni_ficticio': f'DNI-SIM-{i:07d}',
        'email_ficticio': f'cliente{i:06d}@example.com', 'fecha_nacimiento': fecha_nacimiento,
        'provincia': str(rng.choice(PROVINCIAS)), 'fecha_alta_banco': alta,
        'fecha_baja_banco': baja, 'situacion_laboral': ocupacion,
        'fecha_inicio_actividad_laboral': inicio_empleo,
        'ingreso_mensual_declarado_eur': ingreso_base,
        'cantidad_cuentas': num_cuentas, 'cantidad_tarjetas_debito': n_debito,
        'cantidad_tarjetas_credito': n_credito,
        'consentimiento_marketing': consentimiento,
        'canal_preferido': str(rng.choice(['APP_MOVIL','BANCA_WEB','OFICINA'], p=[.65,.24,.11]))
    })
    perfil_por_cliente[cli] = {
        'fecha_alta': alta, 'fecha_baja': baja, 'empleo': ocupacion,
        'fecha_inicio_empleo': inicio_empleo, 'ingreso_base': ingreso_base,
        'riesgo_latente': riesgo_latente, 'volatilidad': volatilidad,
        'cuota_externa_eur': round(float(ingreso_base * rng.uniform(0, .26)), 2),
        'consentimiento': consentimiento,
    }
print('#1 Clientes, cuentas y tarjetas generados')

#1 Clientes, cuentas y tarjetas generados


Ahora, vamos a generar las operaciones financieras de los clientes entre $2023$ y $2026$, incluyendo ingresos, gastos, transferencias, pagos con tarjetas y accesos digitales. También generaremos información sobre reclamaciones, satisfacción y campañas comerciales.

In [8]:
for mes in MESES:
    en_2026 = ESCENARIO_DRIFT_2026 and mes >= pd.Timestamp('2026-01-01')
    for cli, perfil in perfil_por_cliente.items():
        if not es_cliente_activo(perfil, mes):
            continue
        ctas = cuentas_por_cliente[cli]
        principal = ctas[0]
        estres = perfil['riesgo_latente']
        renta = perfil['ingreso_base']
        ingreso_mes = round(float(max(renta * .22, renta * (1 + rng.normal(0, perfil['volatilidad'])))), 2)
        if en_2026:
            ingreso_mes = round(ingreso_mes * 1.025, 2)
        cat_renta = 'NOMINA' if perfil['empleo'] == 'ASALARIADO' else ('PENSION' if perfil['empleo'] == 'PENSIONISTA' else 'INGRESO_ACTIVIDAD')
        ingreso_real = registrar_transaccion(cli, principal, mes + pd.Timedelta(days=2),
                                            'INGRESO', cat_renta, ingreso_mes)
        for secundaria in ctas[1:]:
            traspasar(cli, principal, secundaria, mes + pd.Timedelta(days=5),
                      min(ingreso_mes * rng.uniform(.015, .08), saldo_por_cuenta[principal] * .15))
        n_gastos = int(rng.integers(5, 12))
        factor_gasto = float(np.clip(.46 + .36*estres + rng.normal(0, .09), .34, 1.12))
        gasto_planeado = ingreso_mes * factor_gasto * (1.13 if en_2026 else 1.0)
        reparto = rng.dirichlet(np.ones(n_gastos)*1.7)
        gasto_real = 0.0
        for importe, d in zip(gasto_planeado * reparto, np.sort(rng.integers(7, 25, size=n_gastos))):
            cta = principal if len(ctas) == 1 or rng.random() < .82 else str(rng.choice(ctas[1:]))
            if saldo_por_cuenta[cta] < importe:
                cta = max(ctas, key=lambda c: saldo_por_cuenta[c])
            gasto_real += registrar_transaccion(cli, cta, mes + pd.Timedelta(days=int(d)),
                                                'EGRESO', str(rng.choice(CATEGORIAS_GASTOS)), importe)
        pago_tarjetas = 0.0
        limite_total = sum(l for _, l in tarjetas_credito_por_cliente[cli])
        for tid, limite in tarjetas_credito_por_cliente[cli]:
            compra = round(min(max(0, limite-deuda_por_tarjeta[tid]), ingreso_mes * float(rng.uniform(.02,.16)) * (.7+estres)), 2)
            if compra > 0:
                deuda_por_tarjeta[tid] = round(deuda_por_tarjeta[tid] + compra, 2)
                registrar_mov_tarjeta(cli, tid, mes+pd.Timedelta(days=11), 'COMPRA_CREDITO', compra)
            pago_deseado = min(deuda_por_tarjeta[tid], ingreso_mes*float(rng.uniform(.025,.14))*(1.3-estres))
            cuenta_pagadora = max(ctas, key=lambda c: saldo_por_cuenta[c])
            pago = registrar_transaccion(cli, cuenta_pagadora, mes+pd.Timedelta(days=26),
                                         'EGRESO', 'PAGO_TARJETA_CREDITO', pago_deseado)
            pago_tarjetas += pago
            if pago:
                deuda_por_tarjeta[tid] = round(max(0, deuda_por_tarjeta[tid]-pago),2)
                registrar_mov_tarjeta(cli, tid, mes+pd.Timedelta(days=26), 'PAGO_CREDITO', pago)
            estados_tarjetas.append({
                'cliente_id': cli, 'tarjeta_id': tid, 'mes': mes.strftime('%Y-%m'),
                'limite_credito_eur': limite, 'saldo_utilizado_cierre_eur': deuda_por_tarjeta[tid],
                'utilizacion_cierre': round(deuda_por_tarjeta[tid]/limite,4),
                'compra_mes_eur': compra, 'pago_mes_eur': pago
            })
        saldo_total = 0.0
        for cta in ctas:
            saldo_total += saldo_por_cuenta[cta]
            saldos_mensuales.append({'cliente_id': cli, 'cuenta_id': cta,
                                     'mes': mes.strftime('%Y-%m'), 'saldo_cierre_eur': saldo_por_cuenta[cta]})
        accesos_mes, accesos_fallidos = int(rng.integers(2, 8)) + (2 if en_2026 else 0), 0
        for _ in range(accesos_mes):
            num_acc += 1
            fallo = rng.random() < (.030 if en_2026 else .012)
            accesos_fallidos += int(fallo)
            accesos.append({
                'acceso_id': f'ACC{num_acc:09d}', 'cliente_id': cli,
                'cuenta_id': str(rng.choice(ctas)),
                'fecha_hora': mes+pd.Timedelta(days=int(rng.integers(1, 28)), hours=int(rng.integers(7,23))),
                'canal': str(rng.choice(['APP_MOVIL','BANCA_WEB'], p=[.85,.15] if en_2026 else [.72,.28])),
                'resultado': 'FALLIDO' if fallo else 'EXITOSO'
            })
        deuda_credito = sum(deuda_por_tarjeta[tid] for tid,_ in tarjetas_credito_por_cliente[cli])
        panel_cliente_mensual.append({
            'cliente_id':cli, 'mes':mes.strftime('%Y-%m'), 'ingresos_eur':round(ingreso_real,2),
            'gastos_ordinarios_eur':round(gasto_real,2), 'pagos_tarjetas_eur':round(pago_tarjetas,2),
            'saldo_total_cierre_eur':round(saldo_total,2), 'credito_utilizado_cierre_eur':round(deuda_credito,2),
            'limite_credito_total_eur':round(limite_total,2), 'accesos_digitales':accesos_mes,
            'accesos_fallidos':accesos_fallidos
        })

        # CRM: una reclamacion o consulta puede medirse por SLA, canal y satisfaccion.
        if rng.random() < (.05 + .10*estres + (.03 if en_2026 else 0)):
            n_caso = len(interacciones_crm)+1
            f_apertura = mes+pd.Timedelta(days=int(rng.integers(2,23)))
            dias_respuesta = int(rng.integers(1, 13))
            es_reclamacion = rng.random() < (.45 + .20*estres)
            fecha_resolucion_prevista = f_apertura + pd.Timedelta(days=dias_respuesta)
            resuelto_al_corte = fecha_resolucion_prevista <= FECHA_CORTE_RESULTADOS
            interacciones_crm.append({
                'caso_id':f'CAS{n_caso:08d}', 'cliente_id':cli, 'fecha_apertura':f_apertura,
                'fecha_resolucion':fecha_resolucion_prevista if resuelto_al_corte else pd.NaT,
                'tipo_caso':'RECLAMACION' if es_reclamacion else 'CONSULTA',
                'canal':str(rng.choice(['APP_MOVIL','BANCA_WEB','TELEFONO','OFICINA'])),
                'dias_resolucion':dias_respuesta if resuelto_al_corte else np.nan,
                'sla_objetivo_dias':5,
                'cumple_sla':(dias_respuesta<=5) if resuelto_al_corte else pd.NA,
                'resuelta':bool(resuelto_al_corte),
                'satisfaccion_post_atencion_1_5':int(np.clip(round(5-dias_respuesta/5+rng.normal(0,.6)),1,5)) if resuelto_al_corte else pd.NA
            })
        if rng.random() < .095:
            nps = int(np.clip(round(8.6 - 3*estres + rng.normal(0,1.65) - (1 if en_2026 else 0)),0,10))
            encuestas.append({'encuesta_id':f'ENC{len(encuestas)+1:08d}', 'cliente_id':cli,
                              'fecha':mes+pd.Timedelta(days=27), 'nps_0_10':nps,
                              'csat_1_5':int(np.clip(round((nps+1)/2),1,5)),
                              'respondio':True})
        if perfil['consentimiento'] and rng.random() < .06:
            abierto = bool(rng.random() < (.55-.12*estres))
            conversion = bool(abierto and rng.random() < (.16-.05*estres))
            contactos_campana.append({
                'contacto_id':f'CMP{len(contactos_campana)+1:09d}', 'cliente_id':cli,
                'fecha_contacto':mes+pd.Timedelta(days=16),
                'campana':'VINCULACION_DIGITAL' if en_2026 else 'FIDELIZACION',
                'canal':'EMAIL', 'abierto':abierto, 'conversion':conversion,
                'coste_contacto_eur':0.12, 'ingreso_atribuido_estimado_eur':20.0 if conversion else 0.0
            })
print('#2 Movimientos, accesos, panel mensual y eventos CRM generados')

#2 Movimientos, accesos, panel mensual y eventos CRM generados


Luego, generamos solicitudes de préstamos personales y se evalua mediante una política de aprobación basada en criterios usando ingresos mínimos, capacidad de pago, endeudamiento e historial de morosidad.

Cada solicitud recibe una decisión de aprobación o rechazo, registrándose los motivos correspondientes y las variables financieras utilizadas para justificarla.

In [9]:
panel_df = pd.DataFrame(panel_cliente_mensual)
panel_df['mes_fecha'] = pd.to_datetime(panel_df['mes']+'-01')
panel_por_cliente = {cli:g.sort_values('mes_fecha') for cli,g in panel_df.groupby('cliente_id')}

for cli, perfil in perfil_por_cliente.items():
    if rng.random() >= .82:
        continue
    fecha = pd.Timestamp(rng.choice(FECHAS_COHORTES))
    if not es_cliente_activo(perfil, fecha):
        continue
    prev = panel_por_cliente[cli]
    prev = prev[(prev['mes_fecha'] >= fecha-pd.DateOffset(months=6)) & (prev['mes_fecha'] < fecha)]
    n_meses = len(prev)
    if not n_meses:
        continue
    ingreso_6m = round(float(prev['ingresos_eur'].mean()),2)
    gastos_6m = round(float(prev['gastos_ordinarios_eur'].mean()),2)
    saldo_previo = float(prev.iloc[-1]['saldo_total_cierre_eur'])
    utilizado_previo = float(prev.iloc[-1]['credito_utilizado_cierre_eur'])
    limite_previo = float(prev.iloc[-1]['limite_credito_total_eur'])
    usos_digitales_6m = int(prev['accesos_digitales'].sum())
    cv_ingresos = float(prev['ingresos_eur'].std(ddof=0) / max(ingreso_6m,1))
    cuota_externa = perfil['cuota_externa_eur']
    deuda_externa = round(cuota_externa * float(rng.uniform(7, 25)),2)
    prob_mora = np.array([.83,.10,.05,.015,.005]) * (1 + perfil['riesgo_latente']*np.array([-.55,.15,1.0,2.5,3.0]))
    prob_mora /= prob_mora.sum()
    max_mora_previa = int(rng.choice([0,15,30,60,90], p=prob_mora))
    historial_crediticio.append({
        'cliente_id':cli, 'fecha_consulta':fecha, 'deuda_externa_estimada_eur':deuda_externa,
        'cuota_deuda_externa_mensual_eur':cuota_externa, 'max_dias_mora_12m_previos':max_mora_previa,
        'origen':'CENTRAL_RIESGOS_FICTICIA'
    })
    importe = round(float(np.clip(ingreso_6m * rng.uniform(1.5, 14),
                                   POLITICA['importe_min_eur'], POLITICA['importe_max_eur'])), 2)
    plazo = int(rng.choice(POLITICA['plazos_meses']))
    tin_simulada = round(float(rng.uniform(.055,.135)),4)  # No es una tarifa BBVA.
    cuota_nueva = cuota_francesa(importe,tin_simulada,plazo)
    cuota_min_credito = round(.03*utilizado_previo,2)  # Proxy didactico, no cuota contractual.
    dti = (cuota_externa+cuota_min_credito+cuota_nueva)/max(ingreso_6m,1)
    margen = ingreso_6m-gastos_6m-cuota_externa-cuota_min_credito-cuota_nueva
    antiguedad_lab_meses = max(0,(fecha.year-perfil['fecha_inicio_empleo'].year)*12 + (fecha.month-perfil['fecha_inicio_empleo'].month))

    motivos = []
    if ingreso_6m < POLITICA['ingreso_minimo_eur']:
        motivos.append('INGRESOS_INSUFICIENTES')
    if antiguedad_lab_meses < POLITICA['antiguedad_laboral_minima_meses'] and perfil['empleo']!='PENSIONISTA':
        motivos.append('ANTIGUEDAD_LABORAL_INSUFICIENTE')
    if max_mora_previa >= POLITICA['mora_previa_rechazo_dias']:
        motivos.append('MORA_PREVIA_ELEVADA')
    if dti > POLITICA['dtimax']:
        motivos.append('ENDEUDAMIENTO_EXCESIVO')
    if margen < POLITICA['margen_libre_minimo_eur']:
        motivos.append('MARGEN_DISPONIBLE_INSUFICIENTE')
    if n_meses < POLITICA['historial_bancario_minimo_meses']:
        motivos.append('HISTORIAL_BANCARIO_INSUFICIENTE')
    if not (POLITICA['importe_min_eur']<=importe<=POLITICA['importe_max_eur'] and plazo in POLITICA['plazos_meses']):
        motivos.append('PRODUCTO_FUERA_DE_RANGO')
    aprobado = not motivos
    solicitud_id = f'SOL{len(solicitudes)+1:07d}'
    datos_congelados = {
        'solicitud_id':solicitud_id, 'cliente_id':cli, 'fecha_snapshot':fecha,
        'ingresos_medios_6m_eur':ingreso_6m, 'gastos_ordinarios_medios_6m_eur':gastos_6m,
        'variabilidad_ingresos_6m':round(cv_ingresos,4),
        'saldo_total_ultimo_mes_eur':round(saldo_previo,2),
        'limite_credito_total_ultimo_mes_eur':round(limite_previo,2),
        'credito_utilizado_ultimo_mes_eur':round(utilizado_previo,2),
        'utilizacion_credito_previa':round(utilizado_previo/max(limite_previo,1),4),
        'cuota_deuda_externa_eur':cuota_externa,
        'mora_previa_max_dias':max_mora_previa,
        'cuota_min_tarjeta_estimada_eur':cuota_min_credito,
        'dti_con_nueva_cuota':round(float(dti),4),
        'margen_libre_tras_prestamo_eur':round(margen,2),
        'accesos_digitales_6m':usos_digitales_6m,
        'antiguedad_laboral_meses':antiguedad_lab_meses,
        'meses_historia_disponibles_6m':n_meses,
        'situacion_laboral':perfil['empleo'],
        'importe_solicitado_eur':importe, 'plazo_meses':plazo,
        'cuota_propuesta_eur':cuota_nueva,
        'tasa_nominal_anual_simulada':tin_simulada,
        'particion_temporal':particion_temporal(fecha),
    }
    snapshots.append(datos_congelados)
    solicitudes.append({
        'solicitud_id':solicitud_id, 'cliente_id':cli, 'fecha_solicitud':fecha,
        'importe_solicitado_eur':importe, 'plazo_meses':plazo,
        'tasa_nominal_anual_simulada':tin_simulada, 'cuota_mensual_eur':cuota_nueva,
        'decision_simulada':'APROBADA' if aprobado else 'RECHAZADA',
        'motivos_rechazo':'|'.join(motivos) if motivos else '',
        'numero_motivos_rechazo':len(motivos), 'politica_id':POLITICA['id'],
        'version_politica':POLITICA['version'], 'fecha_horizonte_12m':fecha+pd.DateOffset(months=12),
        'particion_temporal':particion_temporal(fecha), 'incumplimiento_90d_12m':pd.NA,
        'etiqueta_madura':False
    })
print('#3 Solicitudes evaluadas con politica versionada y motivos trazables')

#3 Solicitudes evaluadas con politica versionada y motivos trazables


Ahora, vamos a simular el comportamiento de pago posterior de los clientes cuyos préstamos fueron aprobados, incluyendo fechas de vencimiento, importes pagados y posibles retrasos. A partir de estos registros se identifica el incumplimiento crediticio cuando se alcanzan $90$ días de mora, asignando una etiqueta definitiva únicamente a los préstamos que han completado su período de seguimiento de $12$ meses.

In [10]:
features_por_id = {x['solicitud_id']:x for x in snapshots}
for solicitud in solicitudes:
    if solicitud['decision_simulada'] != 'APROBADA':
        continue
    x = features_por_id[solicitud['solicitud_id']]
    perfil = perfil_por_cliente[solicitud['cliente_id']]
    # Probabilidad generadora.
    riesgo_logit = (-3.9 + 3.5*x['dti_con_nueva_cuota']
                    + 2.1*x['utilizacion_credito_previa']
                    + 1.8*max(0, x['gastos_ordinarios_medios_6m_eur']/max(x['ingresos_medios_6m_eur'],1)-.60)
                    + 1.1*(x['mora_previa_max_dias']>=30)
                    + .5*(perfil['empleo'] in ('AUTONOMO','TEMPORAL'))
                    + .75*(x['saldo_total_ultimo_mes_eur'] < x['ingresos_medios_6m_eur']*.5))
    p_defecto = 1/(1+math.exp(-riesgo_logit))
    impago_futuro = bool(rng.random()<p_defecto)
    primera_cuota_impagada = int(rng.integers(1,8)) if impago_futuro else None
    fecha_inicio = pd.Timestamp(solicitud['fecha_solicitud'])
    fecha_fin_horizonte = pd.Timestamp(solicitud['fecha_horizonte_12m'])
    fecha_obs = min(FECHA_CORTE_RESULTADOS, fecha_fin_horizonte)
    maduro = FECHA_CORTE_RESULTADOS >= fecha_fin_horizonte
    dias_cuotas_solicitud = []
    for k in range(1,13):
        vencimiento = fecha_inicio + pd.DateOffset(months=k)
        if vencimiento > fecha_obs:
            break
        monto_programado = solicitud['cuota_mensual_eur']
        es_impago = primera_cuota_impagada is not None and k >= primera_cuota_impagada
        if es_impago:
            fecha_pago, pagado = pd.NaT, 0.0
        else:
            demora = int(rng.choice([0,1,5,14,30,60],p=[.50,.17,.16,.10,.05,.02]))
            tentativa = vencimiento+pd.Timedelta(days=demora)
            fecha_pago = tentativa if tentativa<=fecha_obs else pd.NaT
            pagado = monto_programado if pd.notna(fecha_pago) else 0.0
        dias_mora = (max(0,(fecha_pago-vencimiento).days) if pd.notna(fecha_pago)
                     else max(0,(fecha_obs-vencimiento).days))
        dias_cuotas_solicitud.append(dias_mora)
        pagos_prestamos.append({
            'solicitud_id':solicitud['solicitud_id'], 'cliente_id':solicitud['cliente_id'],
            'numero_cuota':k, 'fecha_vencimiento':vencimiento,
            'cuota_programada_eur':monto_programado, 'fecha_pago':fecha_pago,
            'importe_pagado_eur':pagado, 'dias_mora_observados':dias_mora,
            'fecha_corte_observacion':fecha_obs,
            'estado_al_corte':('PAGADA' if pagado else ('MORA_90' if dias_mora>=90 else 'VENCIDA'))
        })
    solicitud['etiqueta_madura'] = bool(maduro)
    if maduro:
        solicitud['incumplimiento_90d_12m'] = int(max(dias_cuotas_solicitud,default=0)>=90)
print('#4 Pagos y etiquetas maduras construidos (rechazadas y recientes sin etiqueta)')



#4 Pagos y etiquetas maduras construidos (rechazadas y recientes sin etiqueta)


Luego, calculamos los indicadores mensuales relacionados con las cuatro perspectivas del Balanced Scorecard:
- financiera,
- clientes,
- procesos internos y
- aprendizaje y crecimiento.

Entre ellos se incluyen ingresos y costes operativos estimados, satisfacción del cliente, tiempos de resolución de reclamaciones, formación del personal y automatización de procesos.

In [11]:
casos_df = pd.DataFrame(interacciones_crm)
encuestas_df = pd.DataFrame(encuestas)
contactos_df = pd.DataFrame(contactos_campana)
for mes in MESES:
    inicio = mes
    fin = mes + pd.DateOffset(months=1)
    activos = sum(es_cliente_activo(p, inicio) for p in perfil_por_cliente.values())
    altas = sum(inicio <= p['fecha_alta'] < fin for p in perfil_por_cliente.values())
    bajas = sum(pd.notna(p['fecha_baja']) and inicio <= p['fecha_baja'] < fin for p in perfil_por_cliente.values())
    panel_mes = panel_df[panel_df['mes_fecha']==mes]
    casos_mes = casos_df[(casos_df['fecha_apertura']>=inicio)&(casos_df['fecha_apertura']<fin)]
    enc_mes = encuestas_df[(encuestas_df['fecha']>=inicio)&(encuestas_df['fecha']<fin)]
    contactos_mes = contactos_df[(contactos_df['fecha_contacto']>=inicio)&(contactos_df['fecha_contacto']<fin)]
    # La rentabilidad es un proxy simplificado, no una cuenta de resultados bancaria.
    ingresos_estimados = round(float(panel_mes['credito_utilizado_cierre_eur'].sum()*.015/12 + activos*2.80),2)
    coste_estimado = round(float(activos*1.65 + len(casos_mes)*9 + len(contactos_mes)*.12 + rng.uniform(800,1500)),2)
    n_promotores = int((enc_mes['nps_0_10']>=9).sum())
    n_detractores = int((enc_mes['nps_0_10']<=6).sum())
    nps = round(100*(n_promotores-n_detractores)/len(enc_mes),2) if len(enc_mes) else np.nan
    metricas_operativas.append({
        'mes':mes.strftime('%Y-%m'), 'clientes_activos':activos,
        'altas_clientes':altas, 'bajas_clientes':bajas,
        'ingresos_operativos_estimados_eur':ingresos_estimados,
        'costes_operativos_estimados_eur':coste_estimado,
        'margen_operativo_estimado_eur':round(ingresos_estimados-coste_estimado,2),
        'nps_encuestas':nps, 'n_encuestas':len(enc_mes),
        'casos_abiertos':len(casos_mes),
        'casos_cumplen_sla':int(casos_mes['cumple_sla'].sum()) if len(casos_mes) else 0,
        'tiempo_medio_resolucion_dias':round(float(casos_mes['dias_resolucion'].mean()),2) if len(casos_mes) else np.nan,
        'campanas_contactos':len(contactos_mes),
        'campanas_conversiones':int(contactos_mes['conversion'].sum()) if len(contactos_mes) else 0,
        'operaciones_digitales':int(panel_mes['accesos_digitales'].sum()),
        'accesos_fallidos':int(panel_mes['accesos_fallidos'].sum()),
        'empleados_simulados':int(np.ceil(max(activos,1)/70)),
        'horas_formacion_total':int(rng.integers(24,85)),
        'despliegues_software_simulados':int(rng.integers(1,9)),
        'incidentes_tecnologicos_simulados':int(rng.poisson(.65)),
        'tasa_automatizacion_procesos_simulada':round(float(min(.94,.60+(mes.year-2023)*.075+rng.uniform(-.02,.02))),4)
    })
print('#5 Metrica mensual para las cuatro perspectivas BSC generada')



#5 Metrica mensual para las cuatro perspectivas BSC generada


A continuación, realizamos comprobaciones para garantizar la coherencia de la información generada, verificando identificadores únicos, relaciones entre tablas, saldos y conciliación de transferencias. También comprobamos que las decisiones crediticias respeten los criterios establecidos y que las variables predictoras no incorporen información posterior a la solicitud del préstamo.

In [12]:
df_clientes = pd.DataFrame(clientes)
df_cuentas = pd.DataFrame(cuentas)
df_cuentas['saldo_al_corte_eur'] = df_cuentas['cuenta_id'].map(saldo_por_cuenta)
df_tarjetas = pd.DataFrame(tarjetas)
df_tarjetas['saldo_credito_al_corte_eur'] = df_tarjetas['tarjeta_id'].map(deuda_por_tarjeta)
df_tx = pd.DataFrame(transacciones)
df_accesos = pd.DataFrame(accesos)
df_solicitudes = pd.DataFrame(solicitudes)
df_snapshot = pd.DataFrame(snapshots)
df_pagos = pd.DataFrame(pagos_prestamos)
df_historico = pd.DataFrame(historial_crediticio)
df_saldos = pd.DataFrame(saldos_mensuales)
df_operaciones = pd.DataFrame(metricas_operativas)

assert df_clientes['cliente_id'].is_unique
assert df_cuentas['cuenta_id'].is_unique
assert df_tarjetas['tarjeta_id'].is_unique
assert df_tx['transaccion_id'].is_unique
assert df_solicitudes['solicitud_id'].is_unique
assert df_snapshot['solicitud_id'].is_unique
assert (df_cuentas['saldo_al_corte_eur'] >= -0.0001).all()
assert df_tx['fecha'].max() <= FECHA_FIN_HISTORIA
assert set(df_solicitudes['cliente_id']).issubset(set(df_clientes['cliente_id']))
assert set(df_tx['cuenta_id']).issubset(set(df_cuentas['cuenta_id']))
assert df_snapshot['fecha_snapshot'].equals(df_solicitudes['fecha_solicitud'])
assert (df_snapshot['meses_historia_disponibles_6m'] <= 6).all()
assert df_solicitudes.loc[df_solicitudes['decision_simulada']=='RECHAZADA','incumplimiento_90d_12m'].isna().all()
assert df_solicitudes.loc[~df_solicitudes['etiqueta_madura'],'incumplimiento_90d_12m'].isna().all()
assert (df_solicitudes.loc[df_solicitudes['decision_simulada']=='APROBADA','numero_motivos_rechazo']==0).all()
assert (df_solicitudes.loc[df_solicitudes['decision_simulada']=='RECHAZADA','numero_motivos_rechazo']>0).all()
assert (df_solicitudes.loc[df_solicitudes['etiqueta_madura'],'fecha_horizonte_12m']<=FECHA_CORTE_RESULTADOS).all()
assert set(df_pagos['solicitud_id']).issubset(set(df_solicitudes['solicitud_id']))
assert (df_pagos['fecha_vencimiento']<=df_pagos['fecha_corte_observacion']).all()
# Validacion de sumas de dos lados de transferencias; se ignoran las transferencias sin saldo.
trf = df_tx[df_tx['traspaso_id'].notna()]
if len(trf):
    concilia = trf.assign(signed=np.where(trf['direccion']=='ENTRADA',trf['importe_eur'],-trf['importe_eur']))
    assert np.allclose(concilia.groupby('traspaso_id')['signed'].sum(),0,atol=.01)
# Los ingresos y gastos del panel representan flujos externos, no transferencias ni pagos de tarjeta.
tx_externos = df_tx[df_tx['tipo']=='INGRESO'].copy()
assert (tx_externos['importe_eur']>=0).all()
print('#6 Integridad, conciliacion, aprobacion y madurez de etiquetas verificadas')


#6 Integridad, conciliacion, aprobacion y madurez de etiquetas verificadas


A continuación, organizamos y exportamos las tablas generadas en archivos CSV, junto con un diccionario de datos, metadatos y registros de las políticas de aprobación utilizadas. Además, se crean particiones temporales y mecanismos de verificación de integridad que facilitarán el versionado de datos y su integración en procesos automatizados de entrenamiento y despliegue de modelos.

In [13]:
TABLAS = {
    'clientes':df_clientes, 'cuentas':df_cuentas,
    'tarjetas':df_tarjetas, 'transacciones':df_tx, 'accesos':df_accesos,
    'saldos_mensuales':df_saldos,
    'estados_tarjetas_credito_mensuales':pd.DataFrame(estados_tarjetas),
    'movimientos_tarjetas_credito':pd.DataFrame(movimientos_tarjetas),
    'panel_cliente_mensual':panel_df.drop(columns='mes_fecha'),
    'interacciones_crm':casos_df, 'encuestas_satisfaccion':encuestas_df,
    'contactos_campana':contactos_df,
    'historial_crediticio_previo':df_historico,
    'solicitudes_prestamos':df_solicitudes,
    'features_solicitud':df_snapshot,
    'pagos_prestamos_12m':df_pagos,
    'metricas_operativas_mensuales':df_operaciones,
}
for nombre, df in TABLAS.items():
    df.to_csv(CARPETA_SALIDA/f'{nombre}.csv',index=False,encoding='utf-8-sig')

metadatos = {
    'banco_referencia':BANCO_REFERENCIA,
    'disclaimer':'Simulacion academica. Sin vinculo, datos internos, politicas reales ni representacion de BBVA.',
    'version_dataset':'2.0.0', 'generado_con_semilla':SEMILLA, 'moneda':MONEDA,
    'n_clientes':N_CLIENTES, 'fecha_inicio_historia':str(FECHA_INICIO.date()),
    'fecha_fin_historia':str(FECHA_FIN_HISTORIA.date()),
    'fecha_corte_resultados':str(FECHA_CORTE_RESULTADOS.date()),
    'escenario_drift_gastos_y_digital_2026':ESCENARIO_DRIFT_2026,
    'cohortes_solicitud':[str(f.date()) for f in FECHAS_COHORTES],
    'politica_simulada':POLITICA,
    'definicion_target':'90+ dias de mora durante 12 meses desde solicitud; solo en aprobados con horizonte cumplido.',
    'feature_store':'features_solicitud.csv contiene un snapshot previo al credito por solicitud.',
    'etiquetado':'Rechazadas y solicitudes no maduras no tienen target conocido.',
    'limitaciones':[
        'La decision es una regla hipotetica y no reproduce aprobaciones reales del banco.',
        'Las PD, las tasas y los NPS son enteramente sinteticos y no sirven para decisiones reales.',
        'Los pagos de prestamos posteriores se simulan aparte y no se contabilizan en el libro de cuentas.',
        'Los ingresos y costes operativos del BSC son proxies sinteticos, no contabilidad auditada.',
        'Los prestamos rechazados tienen etiquetas faltantes y producen sesgo de seleccion.',
        'Los casos de CRM sin resolver al corte conservan fecha y satisfaccion como no observadas.',
        'Los IDs y contactos de ejemplo son ficticios, no validos para verificacion de identidad.',
        'El bloque MLOps genera trazabilidad y particiones; no instala Docker ni configura CI/CD.'
    ],
    'fuentes_de_contexto':[
        'https://www.bbva.es/personas/productos/prestamos.html',
        'https://www.bbva.es/finanzas-vistazo/ef/prestamos/scoring-concesion-prestamo.html'
    ]
}
(CARPETA_SALIDA/'metadatos.json').write_text(json.dumps(metadatos,ensure_ascii=False,indent=2,default=str),encoding='utf-8')

diccionario = []
for nombre, df in TABLAS.items():
    for campo in df.columns:
        descripcion = (
            'Identificador ficticio usado para vincular tablas.' if campo.endswith('_id') else
            'Importe expresado en euros.' if campo.endswith('_eur') else
            'Fecha u hora del evento simulado.' if campo.startswith('fecha') else
            'Indicador o caracteristica sintetica; definicion en el bloque que genera la tabla.'
        )
        diccionario.append({'tabla':nombre,'columna':campo,'tipo_pandas':str(df[campo].dtype),
                            'descripcion':descripcion, 'contiene_datos_reales':False})
pd.DataFrame(diccionario).to_csv(CARPETA_SALIDA/'diccionario_datos.csv',index=False,encoding='utf-8-sig')

# Resumen versionable para pipelines: distribucion por cohortes, tasa observada y cobertura.
seguimiento = df_solicitudes.groupby(['fecha_solicitud','particion_temporal'],dropna=False).agg(
    solicitudes=('solicitud_id','size'),aprobadas=('decision_simulada',lambda s:(s=='APROBADA').sum()),
    etiquetas_maduras=('incumplimiento_90d_12m','count'),
    tasa_impago_observada=('incumplimiento_90d_12m','mean')).reset_index()
seguimiento.to_csv(CARPETA_SALIDA/'resumen_cohortes.csv',index=False,encoding='utf-8-sig')

# Huellas SHA-256 para demostrar integridad entre commits, ejecuciones y despliegues.
sha256 = {}
for archivo in sorted(CARPETA_SALIDA.glob('*')):
    if archivo.is_file():
        sha256[archivo.name] = hashlib.sha256(archivo.read_bytes()).hexdigest()
(CARPETA_SALIDA/'manifiesto_sha256.json').write_text(
    json.dumps({'version_dataset':'2.0.0','archivos':sha256},ensure_ascii=False,indent=2),
    encoding='utf-8')

archivo_zip = CARPETA_SALIDA.parent/'dataset_bbva_sintetico_v2.zip'
with zipfile.ZipFile(archivo_zip,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for archivo in CARPETA_SALIDA.glob('*'):
        z.write(archivo,arcname=archivo.name)
print('#7 Exportados',len(TABLAS),'CSV de datos mas diccionario, cohortes y metadatos.')
print('Destino ZIP:',archivo_zip)
print('\nSolicitudes por decision:\n',df_solicitudes['decision_simulada'].value_counts().to_string())
print('\nEtiquetas conocidas:',df_solicitudes['incumplimiento_90d_12m'].notna().sum())
print('Tasa de impago entre aprobadas maduras:',round(float(df_solicitudes['incumplimiento_90d_12m'].mean()),4))
print('Cohortes:\n',seguimiento.to_string(index=False))
if Path('/content').exists():
    try:
        from google.colab import files
        files.download(str(archivo_zip))
    except ImportError:
        pass


#7 Exportados 17 CSV de datos mas diccionario, cohortes y metadatos.
Destino ZIP: /content/dataset_bbva_sintetico_v2.zip

Solicitudes por decision:
 decision_simulada
RECHAZADA    478
APROBADA     325

Etiquetas conocidas: 245
Tasa de impago entre aprobadas maduras: 0.1184
Cohortes:
 fecha_solicitud                 particion_temporal  solicitudes  aprobadas  etiquetas_maduras tasa_impago_observada
     2024-08-01                              TRAIN           77         27                 27              0.111111
     2024-11-01                              TRAIN          103         53                 53              0.113208
     2025-02-01                              TRAIN          103         39                 39              0.153846
     2025-05-01                         VALIDACION           92         44                 44              0.068182
     2025-08-01                      TEST_TEMPORAL           95         41                 41              0.146341
     2025-11-01    

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Finalmente, presentamos un resumen de los datos generados, mostrando la cantidad de registros por tabla y ejemplos de solicitudes de préstamos con sus respectivas decisiones. Su propósito es facilitar una primera revisión del dataset y comprobar que la información está organizada para desarrollar posteriormente los indicadores, el modelo de scoring y los mecanismos de monitorización.

In [14]:
print('\nNumero de filas por tabla:')
for nombre, df in TABLAS.items():
    print(f'{nombre:38} {len(df):>9,}')
print('\nEjemplo: solicitudes y motivos de rechazo:')
print(df_solicitudes[['solicitud_id','fecha_solicitud','importe_solicitado_eur',
                      'decision_simulada','motivos_rechazo','incumplimiento_90d_12m']].head(8).to_string(index=False))


Numero de filas por tabla:
clientes                                   1,000
cuentas                                    1,397
tarjetas                                   1,957
transacciones                            465,587
accesos                                  218,297
saldos_mensuales                          61,865
estados_tarjetas_credito_mensuales        32,384
movimientos_tarjetas_credito              64,768
panel_cliente_mensual                     44,204
interacciones_crm                          3,896
encuestas_satisfaccion                     4,128
contactos_campana                          1,875
historial_crediticio_previo                  803
solicitudes_prestamos                        803
features_solicitud                           803
pagos_prestamos_12m                        3,426
metricas_operativas_mensuales                 46

Ejemplo: solicitudes y motivos de rechazo:
solicitud_id fecha_solicitud  importe_solicitado_eur decision_simulada                         